# 01 · Swiss roll: experimento completo con β seleccionado

Encoder espectral congelado → dos cabezas afines → VAE. Una semilla por proceso, dos T4.

**Cambios respecto a la versión anterior (revisión del tutor):**
- β se elige con el barrido de 01a (regla sin etiquetas). Si el barrido no existe en esta sesión, se ejecuta aquí (~5–10 min).
- Réplicas independientes: cada semilla cambia la muestra de datos **y** la partición (`resample_data`), así que el oráculo también varía.
- Métricas con nombre explícito: `mse_x` (ambiental por coordenada) ≠ `chart_mse_z` (carta por coordenada).
- Controles: oráculo → enrollado exacto g(Π_Ω ẑ) con cota ⅔·MSE_z; decoder entrenado sobre la carta verdadera y alimentado con ẑ.
- Continuación en m (`*_warm`, filas nuevas en cero) frente a entrenamiento desde cero; VAE inicializado desde el AE (`vae_from_ae`, KL warm-up).
- Reconstrucción, KL y varianzas posteriores registradas por separado; alineamientos rígido / similitud / afín.
- Figura de generación: configuración elegida por **validación**, D(μ) y E_q[D(z)], mismo N que la referencia, distancia a la superficie, cobertura y TV sobre la carta.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## β: barrido (si no existe en esta sesión)

In [ ]:
import time
SELECTION = Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')
if not SELECTION.exists():
    t0 = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'beta_sweep.py'), str(ROOT / 'config_beta_sweep.json'), '--launch'],
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode:
        print(r.stderr[-6000:]); raise RuntimeError('Falló el barrido de beta.')
    print(f'Barrido: {(time.time()-t0)/60:.1f} min')
selection = json.loads(SELECTION.read_text())
print(json.dumps(selection, indent=2))

## Configuración

In [ ]:
cfg = json.loads((ROOT / 'config_swiss.json').read_text())
cfg['beta_factor'] = str(SELECTION)   # o un número fijo, p. ej. 48, para reproducir una corrida
# PILOTO por defecto. PAPER_BUDGET amplía N y épocas; revisa el tiempo antes de activarlo.
PAPER_BUDGET = False
if PAPER_BUDGET:
    cfg.update(n=12000, epochs=200)
    # Con otro N la densidad cambia: repite el barrido con ese N (config_beta_sweep.json, 'n').
# Más pasos de optimización si las curvas de entrenamiento no se estabilizan:
# cfg['min_steps'] = 3000
cfg['out'] = '/kaggle/working/results_swiss'
CONFIG = ROOT / 'active_config.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
print(json.dumps(cfg, indent=2))

## Ejecutar en dos GPU

Estimación con la configuración por defecto (8 presupuestos de modos, 5 modelos + continuación, 100 épocas): ~6–10 min por semilla; 3 semillas en 2 GPU ≈ 15–25 min.

In [ ]:
t0 = time.time()
subprocess.run([sys.executable, str(ROOT / 'launch.py'), str(CONFIG)], check=True)
print(f'{(time.time()-t0)/60:.1f} min')

## Resultados

`summary.csv`: media ± desviación entre réplicas. Las columnas ambientales (`mse_x`) y de carta (`chart_mse_z`, `*_chart_mse`) no se comparan entre sí. Elige configuraciones con `validation_mse_x`; reserva test para la evaluación final.

In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = Path(cfg['out'])
summary = pd.read_csv(RESULTS / 'summary.csv', header=[0, 1], index_col=[0, 1])
display(summary.style.format('{:.3g}', na_rep=''))
print(json.loads((RESULTS / 'figure_selection.json').read_text()))
for name in ['reconstruction', 'oracle_decomposition', 'chart_alignment', 'kl_posterior',
             'training_curves', 'swiss_ensembles']:
    p = RESULTS / f'{name}.png'
    if p.exists():
        print(name); display(Image(filename=str(p)))

## Archivos guardados

Por semilla: `encoder.npz`, `data.npz`, `provenance.json` (β, longitud del kernel, semillas de datos y partición, almacenamiento del encoder), pesos `.pt`, historiales, predicciones NPZ (`prediction`, `prediction_mc`, `prior_samples`). Globales: `metrics.csv`, `summary.csv`, `figure_selection.json` y figuras PNG/PDF.